# 01 · Exploring the payments report

Before writing the points engine, this notebook answers five questions about the raw Playbypoint payments report of a padel club: what is in it, which payments were cancelled, how to identify a player, and how to know **when** someone played.

Privacy: the report contains customer names. Every output below is an aggregate or a program name; no row with personal data is printed.

In [1]:
import pandas as pd

df = pd.read_csv("../data/club_payments.csv", low_memory=False)
df.shape

(15577, 27)

## 1. What kinds of payments are there?

In [2]:
df["category"].value_counts().head(12)

category
Reservation                5651
Open play                  3380
Merchandise                1929
Clinic                      897
Membership                  867
Reversal of Reservation     612
Event                       542
Debit                       424
Tournament                  381
Ball machine                243
Lesson                      240
Reversal of Open play       112
Name: count, dtype: int64

Only **Open play**, **Clinic** and **Lesson** earn points. Every other category is ignored by the engine, except as part of a player's monthly spend.

## 2. Which payments were cancelled?

`value_counts()` hides empty values by default. With `dropna=False`, a large group with no status appears:

In [3]:
df["status"].value_counts(dropna=False)

status
Captured              11825
NaN                    3082
Totally refunded        545
Refunded to credit       86
Partially refunded       39
Name: count, dtype: int64

**Finding: an empty status is not an error.** Almost all open plays without a status were paid with club credits, which never go through a card charge:

In [4]:
open_play = df[df["category"] == "Open play"]
open_play.loc[open_play["status"].isna(), "payment_type"].value_counts()

payment_type
Club Credits    356
Cash              6
Name: count, dtype: int64

In [5]:
share = open_play["status"].isna().mean()
print(f"Open plays with no status: {share:.0%}")

Open plays with no status: 11%


Filtering on `status == "Captured"` would have silently dropped about 11% of open plays, and exactly the ones from customers loyal enough to buy credits. **Decision: rows with no status count.**

### Reversals

Each `Reversal of ...` row points to the payment it cancels through `reversed_id`. What status do those original payments have?

In [6]:
reversals = df[df["category"] == "Reversal of Open play"]
originals = df[df["encoded_id"].isin(reversals["reversed_id"])]
print(f"Reversals: {len(reversals)}  ·  originals found: {len(originals)}")
originals["status"].value_counts(dropna=False)

Reversals: 112  ·  originals found: 112


status
Totally refunded      87
Refunded to credit    25
Name: count, dtype: int64

**Decision:** dropping the statuses `Totally refunded` and `Refunded to credit` removes every cancelled payment, so the reversal rows themselves can be ignored.

## 3. How do we identify a player?

In [7]:
print(f"Distinct names:      {df['name'].nunique():,}")
print(f"Distinct member IDs: {df['member_id'].nunique():,}")
print(f"Rows with a member ID: {df['member_id'].notna().mean():.0%}")

Distinct names:      1,037
Distinct member IDs: 180
Rows with a member ID: 61%


`member_id` only exists for members: just 180 of 1,037 names have one. The report has no phone or email. **Decision:** identify players by normalized name (trimmed, single spaces, lowercase) and anonymize them as `Player 001`. Known risk: two people with the same name would be merged. At Arenna the phone number solves this.

## 4. When did someone play?

`payment_date` is when they paid, not when they played. `date_and_time` would be the session time, but it is only filled for some categories:

In [8]:
(df[df["category"].isin(["Open play", "Clinic", "Lesson"])]
 .groupby("category")["date_and_time"].apply(lambda s: s.notna().mean())
 .rename("share with session time"))

category
Clinic       0.0
Lesson       1.0
Open play    0.0
Name: share with session time, dtype: float64

For open plays and clinics, the session date is written inside the program name:

In [9]:
programs = df.loc[df["category"].isin(["Open play", "Clinic"]), "description"].dropna()
programs.drop_duplicates().head(8).tolist()

['Advanced Open Play (level 4+) - May 8 2025',
 'Intermediate Open Play (Level 3.0+) - May 10 2025',
 'Early Intermediate Clinic (Level 2.0-2.9) - ',
 'Beginner Open Play (Level 1-2.0) - May 10 2025',
 'Early Intermediate Open Play (Level 2.0-3.0) - May 11 2025',
 'Ladies Open Play (Level 2.0-3.5) - ',
 'Early Intermediate Open Play (Level 2.0-3.0) - May 13 2025',
 'Intermediate Open Play (Level 3.0+) - May 14 2025']

In [10]:
date_pattern = r"-\s*([A-Z][a-z]{2} \d{1,2} \d{4})\s*$"
play_date = pd.to_datetime(programs.str.extract(date_pattern)[0], format="%b %d %Y")
print(f"Program names without a date: {play_date.isna().mean():.1%}")

Program names without a date: 7.6%


How far apart are payment and play? (days, play date minus payment date)

In [11]:
paid = pd.to_datetime(df.loc[programs.index, "payment_date"].str[:10])
(play_date - paid).dt.days.describe().round(1)

count    3929.0
mean        2.6
std        10.0
min      -115.0
25%         0.0
50%         1.0
75%         2.0
max       266.0
dtype: float64

Most people pay the same day or one day before playing. **Decision:** take the play date from the program name, and fall back to the payment date when it is missing (8% of these rows before cleaning, 4% after). Open plays and clinics carry no start time, so the off-peak bonus cannot be simulated with this report.

## Summary of cleaning decisions

| Decision | Reason |
| --- | --- |
| Keep only Open play, Clinic and Lesson | The activities that earn points |
| Drop `Totally refunded` and `Refunded to credit` | They are the payments cancelled by reversals |
| Keep rows with no status | They are club-credit payments |
| Drop Guest Player and Junior programs | Not a real person; the prize is an adult membership |
| Identify by normalized name, anonymize | No phone or email in the report |
| Play date from the program name | `date_and_time` is empty for open plays and clinics |

These rules are implemented in [`src/club_source.py`](../src/club_source.py).